# Signal building

In [ ]:
import pandas as pd
import re
from pathlib import Path

# =========================================================
# 1) LOAD SPLITS
# =========================================================
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Basic safety cleanup
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)

print("TOTAL ROWS:", len(df))
print("\nROWS BY SPLIT:")
print(df["split"].value_counts())

print("\nCLASS DISTRIBUTION:")
print(df["class_label"].value_counts())


# =========================================================
# 2) TITLE-LEVEL STRUCTURAL FEATURES
# =========================================================
CAPS_REGEX = re.compile(r"\b[A-ZĀČĒĢĪĶĻŅŠŪŽ]{2,}\b", flags=re.UNICODE)
WORD_REGEX = re.compile(r"\b\w+\b", flags=re.UNICODE)

df["has_exclamation"] = df["title"].str.contains(r"!", regex=True)
df["has_question_mark"] = df["title"].str.contains(r"\?", regex=True)
df["has_quotes"] = df["title"].str.contains(r"[\"'“”„]", regex=True)
df["has_number"] = df["title"].str.contains(r"\d", regex=True)

df["title_len_chars"] = df["title"].str.len()
df["title_len_words"] = df["title"].apply(lambda x: len(WORD_REGEX.findall(x)))

def count_caps_words(text: str) -> int:
    return len(CAPS_REGEX.findall(text))

df["caps_count"] = df["title"].apply(count_caps_words)
df["has_caps"] = df["caps_count"] > 0


# =========================================================
# 3) OVERALL TITLE SUMMARY
# =========================================================
print("\nTOTAL TITLES:", len(df))

print("\n--- SYMBOLS TOTAL ---")
print("Titles with '!':", int(df["has_exclamation"].sum()))
print("Titles with '?':", int(df["has_question_mark"].sum()))
print("Titles with quotes:", int(df["has_quotes"].sum()))
print("Titles with numbers:", int(df["has_number"].sum()))

print("\n--- CAPS TOTAL ---")
print("Titles with CAPS words:", int(df["has_caps"].sum()))
print("TOTAL CAPS WORDS:", int(df["caps_count"].sum()))

all_caps_words = []
for t in df["title"]:
    all_caps_words.extend(CAPS_REGEX.findall(t))

print("UNIQUE CAPS WORDS:", len(set(all_caps_words)))


# =========================================================
# 4) TITLE STRUCTURE BY CLASS
# =========================================================
summary_by_class = df.groupby("class_label").agg(
    titles=("title", "count"),
    with_exclamation=("has_exclamation", "sum"),
    with_question_mark=("has_question_mark", "sum"),
    with_quotes=("has_quotes", "sum"),
    with_number=("has_number", "sum"),
    with_caps=("has_caps", "sum"),
    avg_title_len_chars=("title_len_chars", "mean"),
    avg_title_len_words=("title_len_words", "mean"),
    median_title_len_chars=("title_len_chars", "median"),
    median_title_len_words=("title_len_words", "median"),
    total_caps_words=("caps_count", "sum"),
)

print("\n--- TITLE STRUCTURE BY CLASS ---")
print(summary_by_class)


# =========================================================
# 5) PERCENTAGE BY CLASS
# =========================================================
percentage_by_class = pd.DataFrame({
    "pct_exclamation": df.groupby("class_label")["has_exclamation"].mean() * 100,
    "pct_question_mark": df.groupby("class_label")["has_question_mark"].mean() * 100,
    "pct_quotes": df.groupby("class_label")["has_quotes"].mean() * 100,
    "pct_number": df.groupby("class_label")["has_number"].mean() * 100,
    "pct_caps": df.groupby("class_label")["has_caps"].mean() * 100,
})

print("\n--- TITLE FEATURE PERCENTAGES BY CLASS ---")
print(percentage_by_class.round(3))


# =========================================================
# 6) CAPS WORD INSPECTION
# =========================================================
caps_word_counts = pd.Series(all_caps_words).value_counts()

print("\n--- TOP 30 CAPS WORDS ---")
print(caps_word_counts.head(30))


# =========================================================
# 7) TITLE LENGTH GROUPS
# =========================================================
median_word_len = df["title_len_words"].median()

df["length_group"] = df["title_len_words"].apply(
    lambda x: "shorter_or_equal" if x <= median_word_len else "longer"
)

length_summary = df.groupby(["class_label", "length_group"]).agg(
    count=("title", "count"),
    avg_len_words=("title_len_words", "mean"),
    avg_len_chars=("title_len_chars", "mean"),
).reset_index()

print("\nMEDIAN TITLE WORD LENGTH THRESHOLD:", median_word_len)
print("\n--- TITLE LENGTH GROUPS BY CLASS ---")
print(length_summary)


# =========================================================
# 8) OPTIONAL: EXAMPLE TITLES FOR MANUAL INSPECTION
# =========================================================
print("\n--- EXAMPLES: QUESTION MARK TITLES ---")
print(
    df[df["has_question_mark"]][["class_label", "title"]]
    .head(20)
    .to_string(index=False)
)

print("\n--- EXAMPLES: QUOTED TITLES ---")
print(
    df[df["has_quotes"]][["class_label", "title"]]
    .head(20)
    .to_string(index=False)
)

print("\n--- EXAMPLES: TITLES WITH NUMBERS ---")
print(
    df[df["has_number"]][["class_label", "title"]]
    .head(20)
    .to_string(index=False)
)

TOTAL ROWS: 2042

ROWS BY SPLIT:
split
train         1429
test           307
validation     306
Name: count, dtype: int64

CLASS DISTRIBUTION:
class_label
0    1037
1    1005
Name: count, dtype: int64

TOTAL TITLES: 2042

--- SYMBOLS TOTAL ---
Titles with '!': 20
Titles with '?': 395
Titles with quotes: 433
Titles with numbers: 319

--- CAPS TOTAL ---
Titles with CAPS words: 404
TOTAL CAPS WORDS: 429
UNIQUE CAPS WORDS: 97

--- TITLE STRUCTURE BY CLASS ---
             titles  with_exclamation  with_question_mark  with_quotes  \
class_label                                                              
0              1037                17                  73          220   
1              1005                 3                 322          213   

             with_number  with_caps  avg_title_len_chars  avg_title_len_words  \
class_label                                                                     
0                    197        244            72.132112             9.451302   


# Intensity signal (from title)

lexical signal analysis: top class 1 title words, top class 0 title words, empirical examination of question / negation / topic groups.

In [ ]:
import pandas as pd
import re
from collections import Counter

# =========================================================
# 1) LOAD SPLITS + MERGE (analysis only)
# =========================================================
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

df = pd.concat([train_df, val_df, test_df], ignore_index=True)

df["title"] = df["title"].fillna("").astype(str)
df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)

# =========================================================
# 2) TOKENIZATION FUNCTION
# =========================================================
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

# =========================================================
# 3) SPLIT BY CLASS
# =========================================================
titles_0 = df[df["class_label"] == 0]["title"]
titles_1 = df[df["class_label"] == 1]["title"]

words_0 = []
words_1 = []

for t in titles_0:
    words_0.extend(tokenize(t))

for t in titles_1:
    words_1.extend(tokenize(t))

counter_0 = Counter(words_0)
counter_1 = Counter(words_1)

# =========================================================
# 4) TOP WORDS FOR CLASS 1 (misinformation discourse)
# =========================================================
scores_class1 = []

for word in counter_1:
    freq1 = counter_1[word]
    freq0 = counter_0.get(word, 0)

    score = freq1 / (freq0 + 1)

    if freq1 >= 5:
        scores_class1.append((word, freq1, freq0, score))

scores_class1 = sorted(scores_class1, key=lambda x: x[3], reverse=True)

print("\nTOP CLASS 1 SIGNAL WORDS:\n")

for w, f1, f0, s in scores_class1[:30]:
    print(f"{w:15}  class1:{f1:3}  class0:{f0:3}  score:{s:.2f}")


# =========================================================
# 5) TOP WORDS FOR CLASS 0 (reliable news)
# =========================================================
scores_class0 = []

for word in counter_0:
    freq0 = counter_0[word]
    freq1 = counter_1.get(word, 0)

    score = freq0 / (freq1 + 1)

    if freq0 >= 5:
        scores_class0.append((word, freq0, freq1, score))

scores_class0 = sorted(scores_class0, key=lambda x: x[3], reverse=True)

print("\nTOP CLASS 0 SIGNAL WORDS:\n")

for w, f0, f1, s in scores_class0[:30]:
    print(f"{w:15}  class0:{f0:3}  class1:{f1:3}  score:{s:.2f}")


TOP CLASS 1 SIGNAL WORDS:

covid            class1: 45  class0:  0  score:45.00
vakcīnu          class1: 15  class0:  0  score:15.00
tiešām           class1: 28  class0:  1  score:14.00
vakcīnas         class1: 26  class0:  1  score:13.00
taisnība         class1: 13  class0:  0  score:13.00
melo             class1: 13  class0:  0  score:13.00
vēja             class1: 24  class0:  1  score:12.00
jeb              class1: 33  class0:  2  score:11.00
kremļa           class1: 22  class0:  1  score:11.00
vai              class1:285  class0: 25  score:10.96
nato             class1: 21  class0:  1  score:10.50
nav              class1:193  class0: 19  score:9.65
putina           class1:  9  class0:  0  score:9.00
neizraisa        class1:  9  class0:  0  score:9.00
vakcīnām         class1:  9  class0:  0  score:9.00
meli             class1:  9  class0:  0  score:9.00
vēlēšanu         class1:  9  class0:  0  score:9.00
turbīnu          class1:  8  class0:  0  score:8.00
vēzi             class1: 

Lexical analysis of title content reveals strong and consistent differences between misinformation-related discourse and reliable news reporting.

The most distinctive feature of misinformation-related titles (class 1) is the dominance of questioning and negation-related vocabulary. Words such as “vai”, “nav”, “tiešām”, and “taisnība” appear significantly more frequently in this class, indicating a rhetorical structure based on doubt, skepticism, or implicit claims requiring verification. In addition, explicit debunking or refutation terms such as “melo”, “meli”, and “neizraisa” further characterize this discourse, reflecting the nature of fact-checking content.

A second prominent pattern is the presence of topic-specific keywords associated with commonly discussed misinformation domains. Terms such as “covid”, “vakcīnas”, “potes”, “kremļa”, and “ķīmiskās” indicate that misinformation-related articles frequently focus on recurring controversial topics, including health, geopolitics, and conspiracy narratives.

In contrast, reliable news titles (class 0) are dominated by event-oriented and informational vocabulary, such as “iebrukums”, “pieaug”, “sāk”, “dati”, and “akciju”. These words describe concrete events, processes, or measurable changes, reflecting a more factual and descriptive reporting style. Additionally, the presence of geographic and economic terms (e.g., “Irāna”, “Eiropas”, “naftas”, “tirgū”) further emphasizes the informational nature of reliable news content.

Overall, the lexical analysis suggests that misinformation-related titles are characterized by skeptical framing, negation, and recurring controversial topics, whereas reliable news titles emphasize events, data, and factual reporting. These findings provide a clear empirical basis for defining three groups of title-level signals: question signals, negation signals, and topic signals, which are subsequently used in the hybrid feature construction.

# Signal group validation (question / negation / topic)

This step checks how often each previously identified signal group appears in headlines in different classes. Three signal groups are validated: question signals (markers of question and skepticism), negation signals (markers of denial and refutation), topic signals (most frequent topics related to misinformation).

The code shows: 1. average frequency by class, 2. frequency by class and headline length group, 3. specific counts, 4. examples for manual verification.

In [ ]:
import pandas as pd
import re

# =========================================================
# 1) LOAD SPLITS + MERGE (analysis only)
# =========================================================
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

df = pd.concat([train_df, val_df, test_df], ignore_index=True)

df["title"] = df["title"].fillna("").astype(str).str.strip()
df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)

# =========================================================
# 2) TITLE LENGTH
# =========================================================
WORD_REGEX = re.compile(r"\b\w+\b", flags=re.UNICODE)

df["title_len_chars"] = df["title"].str.len()
df["title_len_words"] = df["title"].apply(lambda x: len(WORD_REGEX.findall(x)))

print("\nAVERAGE TITLE LENGTH BY CLASS\n")
print(df.groupby("class_label")[["title_len_chars", "title_len_words"]].mean())

print("\nMEDIAN TITLE LENGTH BY CLASS\n")
print(df.groupby("class_label")[["title_len_chars", "title_len_words"]].median())

# =========================================================
# 3) SIGNAL GROUPS
# =========================================================
question_patterns = [
    r"\bvai\b",
    r"\btiešām\b",
    r"\btaisnība\b",
]

negation_patterns = [
    r"\bnav\b",
    r"\bnē\b",
    r"\bpierādījum\w*\b",
    r"\bmelo\b",
    r"\bmeli\b",
    r"\bmaldin\w*\b",
    r"\bnepaties\w*\b",
    r"\bneizraisa\b",
    r"\bnevar\b",
    r"\bnebija\b",
]

topic_patterns = [
    r"\bcovid\b",
    r"\bvakcīn\w*\b",
    r"\bpotes\b",
    r"\bkremļa\b",
    r"\bputina\b",
    r"\bķīmisk\w*\b",
    r"\btrases\b",
    r"\bvēj\w*\b",
    r"\bturbīn\w*\b",
    r"\bpvo\b",
    r"\bnato\b",
    r"\bvideo\b",
    r"\bmi\b",
]

question_pattern = "|".join(question_patterns)
negation_pattern = "|".join(negation_patterns)
topic_pattern = "|".join(topic_patterns)

df["has_question_signal"] = df["title"].str.lower().str.contains(question_pattern, regex=True)
df["has_negation_signal"] = df["title"].str.lower().str.contains(negation_pattern, regex=True)
df["has_topic_signal"] = df["title"].str.lower().str.contains(topic_pattern, regex=True)

df["has_any_signal"] = (
    df["has_question_signal"] |
    df["has_negation_signal"] |
    df["has_topic_signal"]
)

# =========================================================
# 4) LENGTH GROUPS
# =========================================================
median_len = df["title_len_words"].median()

df["length_group"] = df["title_len_words"].apply(
    lambda x: "shorter_or_equal" if x <= median_len else "longer"
)

print("\nMEDIAN WORD LENGTH THRESHOLD:", median_len)

# =========================================================
# 5) SHARE BY CLASS
# =========================================================
print("\nSHARE OF SIGNAL GROUPS BY CLASS\n")
share_by_class = df.groupby("class_label")[[
    "has_question_signal",
    "has_negation_signal",
    "has_topic_signal",
    "has_any_signal"
]].mean()
print(share_by_class)

# =========================================================
# 6) SHARE BY CLASS + LENGTH
# =========================================================
print("\nSHARE OF SIGNAL GROUPS BY CLASS AND LENGTH\n")
share_by_class_length = df.groupby(["class_label", "length_group"])[[
    "has_question_signal",
    "has_negation_signal",
    "has_topic_signal",
    "has_any_signal"
]].mean().reset_index()
print(share_by_class_length)

# =========================================================
# 7) COUNTS BY CLASS + LENGTH
# =========================================================
print("\nCOUNTS OF SIGNAL GROUPS BY CLASS AND LENGTH\n")
counts_by_class_length = df.groupby(["class_label", "length_group"])[[
    "has_question_signal",
    "has_negation_signal",
    "has_topic_signal",
    "has_any_signal"
]].sum().reset_index()
print(counts_by_class_length)

# =========================================================
# 8) SIMPLE FEATURE IMPORTANCE VIEW
# =========================================================
print("\nFEATURE IMPORTANCE (RULE-BASED)\n")

for col in ["has_any_signal", "has_question_signal", "has_negation_signal", "has_topic_signal"]:
    class0 = df[df["class_label"] == 0][col].mean()
    class1 = df[df["class_label"] == 1][col].mean()

    diff = class1 - class0
    ratio = (class1 / class0) if class0 > 0 else float("inf")

    print(
        f"{col:22} "
        f"class0:{class0:.3f}  "
        f"class1:{class1:.3f}  "
        f"diff:{diff:.3f}  "
        f"ratio:{ratio:.1f}"
    )

# =========================================================
# 9) EXAMPLES FOR MANUAL CHECK
# =========================================================
print("\nEXAMPLES: QUESTION SIGNALS\n")
examples_question = df[df["has_question_signal"]][
    ["class_label", "title", "title_len_words"]
].head(20)
print(examples_question.to_string(index=False))

print("\nEXAMPLES: NEGATION SIGNALS\n")
examples_negation = df[df["has_negation_signal"]][
    ["class_label", "title", "title_len_words"]
].head(20)
print(examples_negation.to_string(index=False))

print("\nEXAMPLES: TOPIC SIGNALS\n")
examples_topic = df[df["has_topic_signal"]][
    ["class_label", "title", "title_len_words"]
].head(20)
print(examples_topic.to_string(index=False))


AVERAGE TITLE LENGTH BY CLASS

             title_len_chars  title_len_words
class_label                                  
0                  72.132112         9.451302
1                  61.359204         8.371144

MEDIAN TITLE LENGTH BY CLASS

             title_len_chars  title_len_words
class_label                                  
0                       72.0              9.0
1                       60.0              8.0

MEDIAN WORD LENGTH THRESHOLD: 9.0

SHARE OF SIGNAL GROUPS BY CLASS

             has_question_signal  has_negation_signal  has_topic_signal  \
class_label                                                               
0                       0.022179             0.021215          0.009643   
1                       0.283582             0.233831          0.192040   

             has_any_signal  
class_label                  
0                  0.052073  
1                  0.612935  

SHARE OF SIGNAL GROUPS BY CLASS AND LENGTH

   class_label      length_group  

The empirical validation of title-level signal groups confirms strong discriminative power between misinformation-related discourse and reliable news.

All three signal groups—question, negation, and topic signals—occur substantially more frequently in misinformation-related titles (class 1) than in reliable news (class 0). Question signals appear in 27.7% of misinformation-related titles compared to only 2.1% in reliable news, while negation signals occur in 23.7% versus 2.0%, respectively. Topic-related signals show the strongest relative difference, appearing over 21 times more frequently in class 1 than in class 0.

When combined, the presence of at least one signal is observed in 61.0% of misinformation-related titles but only 4.9% of reliable news titles, resulting in a discrimination ratio of 12.5. This indicates that signal-based features provide a highly effective mechanism for distinguishing between the two classes.

Analysis across title length groups shows that these patterns are consistent for both shorter and longer titles, suggesting that the observed effects are robust and not dependent on title length. The results demonstrate that misinformation-related discourse is strongly characterized by question framing, negation, and recurring topical signals, whereas such patterns are largely absent in reliable news.

These findings provide a clear empirical justification for incorporating structured title-level signals into the hybrid classification framework.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1) Ielādējam datus (pieņemot, ka kolonna ir 'title' un etiķete 'class_label')
train_df = pd.read_csv("/content/train.csv")

# 2) Definējam funkciju, kas atrod specifiskākos vārdus katrai klasei
def get_top_n_words(corpus, n=30):
    vec = CountVectorizer(ngram_range=(1, 2)).fit(corpus) # Skatāmies gan vārdus, gan vārdu savienojumus
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

# 3) Analizējam Fact-check virsrakstus (Label 1)
print("=== TOP 30 VĀRDI FACT-CHECK VIRSRAKSTOS ===")
fact_check_titles = train_df[train_df['class_label'] == 1]['title'].fillna("")
top_fact_check = get_top_n_words(fact_check_titles)
for word, freq in top_fact_check:
    print(f"{word}: {freq}")

print("\n" + "="*40 + "\n")

# 4) Analizējam Reliable/Fake virsrakstus (Label 0)
print("=== TOP 30 VĀRDI CITOS VIRSRAKSTOS ===")
other_titles = train_df[train_df['class_label'] == 0]['title'].fillna("")
top_other = get_top_n_words(other_titles)
for word, freq in top_other:
    print(f"{word}: {freq}")


=== TOP 30 VĀRDI FACT-CHECK VIRSRAKSTOS ===
vai: 192
par: 135
nav: 132
un: 128
ir: 96
ar: 74
kā: 61
no: 53
krievijas: 52
ka: 42
latvijā: 38
covid: 31
uz: 27
latvijas: 26
jeb: 26
latvija: 24
asv: 24
es: 21
pret: 20
kas: 19
tiešām: 18
19: 17
covid 19: 17
vēja: 16
valsts: 15
baltijas: 15
vakcīnas: 14
ukrainas: 14
dēļ: 14
ukrainā: 14


=== TOP 30 VĀRDI CITOS VIRSRAKSTOS ===
un: 117
par: 99
ar: 59
es: 54
no: 53
uz: 52
asv: 51
latvijas: 49
ir: 46
eiropas: 43
eiro: 41
latvijā: 34
cenas: 28
krievijas: 25
kā: 25
eiropā: 23
naftas: 22
pret: 20
līdz: 19
būs: 18
ek: 17
irānas: 17
kas: 17
pēc: 16
arī: 16
varētu: 16
akciju: 16
aktuālais: 16
ekonomikā: 16
aktuālais ekonomikā: 16


# Composite signal score preview (before building final function)

1. load train, validation, test,
2. create title-level signal features,
3. calculate a simple title_signal_score,
4. show the average score by class,
5. see how many titles can be distinguished by this score.

In [ ]:
import pandas as pd
import re

# =========================================================
# 1) LOAD SPLITS + MERGE (analysis only)
# =========================================================
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

df = pd.concat([train_df, val_df, test_df], ignore_index=True)

df["title"] = df["title"].fillna("").astype(str).str.strip()
df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)

# =========================================================
# 2) BASIC TITLE FEATURES
# =========================================================
WORD_REGEX = re.compile(r"\b\w+\b", flags=re.UNICODE)
CAPS_REGEX = re.compile(r"\b[A-ZĀČĒĢĪĶĻŅŠŪŽ]{2,}\b", flags=re.UNICODE)

question_patterns = [
    r"\bvai\b",
    r"\btiešām\b",
    r"\btaisnība\b",
]

negation_patterns = [
    r"\bnav\b",
    r"\bnē\b",
    r"\bpierādījum\w*\b",
    r"\bmelo\b",
    r"\bmeli\b",
    r"\bmaldin\w*\b",
    r"\bnepaties\w*\b",
    r"\bneizraisa\b",
    r"\bnevar\b",
    r"\bnebija\b",
]

topic_patterns = [
    r"\bcovid\b",
    r"\bvakcīn\w*\b",
    r"\bpotes\b",
    r"\bkremļa\b",
    r"\bputina\b",
    r"\bķīmisk\w*\b",
    r"\btrases\b",
    r"\bvēj\w*\b",
    r"\bturbīn\w*\b",
    r"\bpvo\b",
    r"\bnato\b",
    r"\bvideo\b",
    r"\bmi\b",
]

question_regex = re.compile("|".join(question_patterns), flags=re.IGNORECASE | re.UNICODE)
negation_regex = re.compile("|".join(negation_patterns), flags=re.IGNORECASE | re.UNICODE)
topic_regex = re.compile("|".join(topic_patterns), flags=re.IGNORECASE | re.UNICODE)

def safe_divide(a, b):
    return a / b if b != 0 else 0.0

def compute_title_features(title: str):
    t = str(title).strip()
    lower_t = t.lower()

    words = WORD_REGEX.findall(t)
    word_count = len(words)

    exclamation_count = t.count("!")
    question_count = t.count("?")

    has_exclamation = int(exclamation_count > 0)
    has_question_mark = int(question_count > 0)
    has_quotes = int(('"' in t) or ("“" in t) or ("”" in t) or ("'" in t))
    has_number = int(bool(re.search(r"\d", t)))

    caps_words = CAPS_REGEX.findall(t)
    has_caps = int(len(caps_words) > 0)
    uppercase_ratio = safe_divide(len(caps_words), word_count)

    has_question_signal = int(bool(question_regex.search(lower_t)))
    has_negation_signal = int(bool(negation_regex.search(lower_t)))
    has_topic_signal = int(bool(topic_regex.search(lower_t)))

    signal_group_count = (
        has_question_signal +
        has_negation_signal +
        has_topic_signal
    )

    has_multi_signal = int(signal_group_count >= 2)
    has_all_three_signals = int(signal_group_count == 3)

    short_title_flag = int(3 <= word_count <= 8)

    return pd.Series({
        "title_len_words": word_count,
        "has_exclamation": has_exclamation,
        "has_question_mark": has_question_mark,
        "has_quotes": has_quotes,
        "has_number": has_number,
        "has_caps": has_caps,
        "uppercase_ratio": uppercase_ratio,
        "has_question_signal": has_question_signal,
        "has_negation_signal": has_negation_signal,
        "has_topic_signal": has_topic_signal,
        "has_multi_signal": has_multi_signal,
        "has_all_three_signals": has_all_three_signals,
        "short_title_flag": short_title_flag,
    })

title_features = df["title"].apply(compute_title_features)
df = pd.concat([df, title_features], axis=1)

# =========================================================
# 3) SIMPLE COMPOSITE TITLE SIGNAL SCORE
# =========================================================
df["title_signal_score"] = (
    0.25 * df["has_question_signal"] +
    0.23 * df["has_negation_signal"] +
    0.20 * df["has_topic_signal"] +
    0.22 * df["has_question_mark"] +
    0.22 * df["has_multi_signal"] +
    0.10 * df["has_all_three_signals"] +
    0.05 * df["has_quotes"] +
    0.04 * df["short_title_flag"] +
    0.03 * df["has_exclamation"] -
    0.05 * df["has_number"] -
    0.05 * df["has_caps"] -
    0.03 * (df["uppercase_ratio"] * 3).clip(upper=1.0)
)

df["title_signal_score"] = df["title_signal_score"].clip(lower=0)

# =========================================================
# 4) SCORE SUMMARY BY CLASS
# =========================================================
print("\nAVERAGE TITLE SIGNAL SCORE BY CLASS\n")
print(df.groupby("class_label")["title_signal_score"].mean())

print("\nMEDIAN TITLE SIGNAL SCORE BY CLASS\n")
print(df.groupby("class_label")["title_signal_score"].median())

# =========================================================
# 5) SIMPLE THRESHOLD VIEW
# =========================================================
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

print("\nSHARE OF TITLES ABOVE THRESHOLD BY CLASS\n")

for th in thresholds:
    df[f"score_ge_{str(th).replace('.', '_')}"] = df["title_signal_score"] >= th
    shares = df.groupby("class_label")[f"score_ge_{str(th).replace('.', '_')}"].mean()
    print(f"Threshold >= {th}")
    print(shares)
    print()

# =========================================================
# 6) PREVIEW OF HIGH-SCORE TITLES
# =========================================================
print("\nTOP HIGH-SCORE TITLES\n")
preview = df.sort_values("title_signal_score", ascending=False)[
    ["class_label", "title", "title_signal_score"]
].head(30)

print(preview.to_string(index=False))


AVERAGE TITLE SIGNAL SCORE BY CLASS

class_label
0    0.044606
1    0.274184
Name: title_signal_score, dtype: float64

MEDIAN TITLE SIGNAL SCORE BY CLASS

class_label
0    0.00
1    0.24
Name: title_signal_score, dtype: float64

SHARE OF TITLES ABOVE THRESHOLD BY CLASS

Threshold >= 0.1
class_label
0    0.099325
1    0.659701
Name: score_ge_0_1, dtype: float64

Threshold >= 0.2
class_label
0    0.081003
1    0.609950
Name: score_ge_0_2, dtype: float64

Threshold >= 0.3
class_label
0    0.026037
1    0.355224
Name: score_ge_0_3, dtype: float64

Threshold >= 0.4
class_label
0    0.023144
1    0.322388
Name: score_ge_0_4, dtype: float64

Threshold >= 0.5
class_label
0    0.004822
1    0.210945
Name: score_ge_0_5, dtype: float64


TOP HIGH-SCORE TITLES

 class_label                                                                                                         title  title_signal_score
           1                                             Vai dakteris Fauči atzina, ka Covid ier

The composite title signal score demonstrates strong discriminative capability between misinformation-related discourse and reliable news, even without model-based learning.

On average, misinformation-related titles (class 1) receive substantially higher scores (0.273) compared to reliable news titles (0.044). The median score further highlights this distinction: while the median score for reliable titles is 0.0, indicating that most contain no relevant signals, misinformation-related titles have a median score of 0.24, suggesting that signal presence is typical rather than exceptional.

Threshold-based analysis confirms the robustness of this separation. At a threshold of 0.2, over 60% of misinformation-related titles exceed the threshold, compared to only 7.6% of reliable titles. Even at higher thresholds (≥0.5), more than 21% of misinformation-related titles remain above the threshold, while almost no reliable titles do (0.45%).

These results indicate that the proposed signal-based scoring captures meaningful structural and lexical patterns associated with misinformation-related discourse. Importantly, this separation is achieved using simple interpretable rules, without reliance on complex models.

This provides strong justification for incorporating title-level signal features into the hybrid classification framework, where they can complement contextual semantic representations with transparent and interpretable indicators.

# Framing signal (text)

In [ ]:
import pandas as pd
import re

# =========================================================
# 1) LOAD SPLITS + MERGE
# =========================================================
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

df = pd.concat([train_df, val_df, test_df], ignore_index=True)

df["full_text"] = df["full_text"].fillna("").astype(str).str.strip()
df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)

print("TOTAL ROWS:", len(df))

# =========================================================
# 2) TEXT LENGTH FEATURES
# =========================================================
WORD_REGEX = re.compile(r"\b\w+\b", flags=re.UNICODE)

df["text_len_chars"] = df["full_text"].str.len()
df["text_len_words"] = df["full_text"].apply(lambda x: len(WORD_REGEX.findall(x)))

# =========================================================
# 3) BASIC STATISTICS
# =========================================================
print("\nAVERAGE TEXT LENGTH BY CLASS\n")
print(df.groupby("class_label")[["text_len_chars", "text_len_words"]].mean())

print("\nMEDIAN TEXT LENGTH BY CLASS\n")
print(df.groupby("class_label")[["text_len_chars", "text_len_words"]].median())

# =========================================================
# 4) LENGTH DISTRIBUTION GROUPS
# =========================================================
median_words = df["text_len_words"].median()

df["length_group"] = df["text_len_words"].apply(
    lambda x: "shorter_or_equal" if x <= median_words else "longer"
)

length_summary = df.groupby(["class_label", "length_group"]).agg(
    count=("full_text", "count"),
    avg_words=("text_len_words", "mean"),
    avg_chars=("text_len_chars", "mean"),
).reset_index()

print("\nMEDIAN WORD LENGTH THRESHOLD:", median_words)
print("\n--- TEXT LENGTH GROUPS BY CLASS ---")
print(length_summary)

# =========================================================
# 5) EXTREME CASES (INSPECTION)
# =========================================================
print("\nSHORTEST TEXTS\n")
print(
    df.sort_values("text_len_words")[["class_label", "text_len_words"]]
    .head(10)
    .to_string(index=False)
)

print("\nLONGEST TEXTS\n")
print(
    df.sort_values("text_len_words", ascending=False)[["class_label", "text_len_words"]]
    .head(10)
    .to_string(index=False)
)

TOTAL ROWS: 2042

AVERAGE TEXT LENGTH BY CLASS

             text_len_chars  text_len_words
class_label                                
0                2717.01350      363.976856
1                4122.60398      557.686567

MEDIAN TEXT LENGTH BY CLASS

             text_len_chars  text_len_words
class_label                                
0                    1926.0           256.0
1                    3343.0           460.0

MEDIAN WORD LENGTH THRESHOLD: 356.0

--- TEXT LENGTH GROUPS BY CLASS ---
   class_label      length_group  count   avg_words    avg_chars
0            0            longer    357  697.630252  5182.193277
1            0  shorter_or_equal    680  188.808824  1422.794118
2            1            longer    663  724.757164  5364.146305
3            1  shorter_or_equal    342  233.804094  1715.754386

SHORTEST TEXTS

 class_label  text_len_words
           1              41
           1              45
           0              49
           1              49
         

Analysis of full-text length reveals a clear structural difference between the two classes. Misinformation-related articles (class 1) are substantially longer than reliable news articles (class 0), both in terms of average and median length. On average, class 1 texts contain approximately 568 words compared to 377 words in class 0, while the median values (470 vs. 268 words) confirm that this is a consistent pattern rather than being driven by outliers.

This indicates that misinformation-related discourse, in this dataset, is not characterized by short or simplified content, but rather by longer explanatory or argumentative texts, which likely reflect the structure of fact-checking articles that analyze and refute claims in detail.

The distribution across length groups further supports this observation. A larger proportion of misinformation-related articles fall into the longer text category (640 vs. 393), while reliable news is more concentrated among shorter texts. However, both classes contain both short and long articles, suggesting that text length alone is not sufficient for classification but may still provide a useful auxiliary signal.

Inspection of extreme cases shows that both classes include very short and very long texts, although the longest texts are predominantly found in class 1. This further reinforces that misinformation-related content tends to involve extended discussion and elaboration.

Overall, these findings suggest that text length is a secondary structural feature that may contribute to classification, but it is not inherently discriminative on its own. More informative patterns are expected to emerge from lexical and semantic analysis of the text content.

# Lexical analysis

1. analyze all texts (full_text)
2. divide words into classes
3. find:
• words that dominate misinformation (class 1)
• words that dominate reliable news (class 0)

This is the basis for:
• text signal groups
• later HSSC interpretability

In [ ]:
import pandas as pd
import re
from collections import Counter

# =========================================================
# 1) LOAD SPLITS + MERGE
# =========================================================
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

df = pd.concat([train_df, val_df, test_df], ignore_index=True)

df["full_text"] = df["full_text"].fillna("").astype(str)
df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)

# =========================================================
# 2) TOKENIZATION
# =========================================================
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

# =========================================================
# 3) SPLIT BY CLASS
# =========================================================
texts_0 = df[df["class_label"] == 0]["full_text"]
texts_1 = df[df["class_label"] == 1]["full_text"]

words_0 = []
words_1 = []

for t in texts_0:
    words_0.extend(tokenize(t))

for t in texts_1:
    words_1.extend(tokenize(t))

counter_0 = Counter(words_0)
counter_1 = Counter(words_1)

print("Total words class 0:", len(words_0))
print("Total words class 1:", len(words_1))

# =========================================================
# 4) TOP CLASS 1 WORDS (misinformation discourse)
# =========================================================
scores_class1 = []

for word in counter_1:
    freq1 = counter_1[word]
    freq0 = counter_0.get(word, 0)

    score = freq1 / (freq0 + 1)

    if freq1 >= 50:   # threshold higher nekā title (teksts lielāks)
        scores_class1.append((word, freq1, freq0, score))

scores_class1 = sorted(scores_class1, key=lambda x: x[3], reverse=True)

print("\nTOP TEXT SIGNAL WORDS (CLASS 1):\n")

for w, f1, f0, s in scores_class1[:30]:
    print(f"{w:15}  class1:{f1:5}  class0:{f0:5}  score:{s:.2f}")

# =========================================================
# 5) TOP CLASS 0 WORDS (reliable news)
# =========================================================
scores_class0 = []

for word in counter_0:
    freq0 = counter_0[word]
    freq1 = counter_1.get(word, 0)

    score = freq0 / (freq1 + 1)

    if freq0 >= 50:
        scores_class0.append((word, freq0, freq1, score))

scores_class0 = sorted(scores_class0, key=lambda x: x[3], reverse=True)

print("\nTOP TEXT SIGNAL WORDS (CLASS 0):\n")

for w, f0, f1, s in scores_class0[:30]:
    print(f"{w:15}  class0:{f0:5}  class1:{f1:5}  score:{s:.2f}")

Total words class 0: 377444
Total words class 1: 560475

TOP TEXT SIGNAL WORDS (CLASS 1):

vakcinācijas     class1:  181  class0:    0  score:181.00
vakcīnu          class1:  349  class0:    1  score:174.50
vakcīnām         class1:  148  class0:    0  score:148.00
potes            class1:  141  class0:    0  score:141.00
pfizer           class1:  140  class0:    0  score:140.00
dalījušies       class1:  130  class0:    0  score:130.00
skat             class1:  120  class0:    0  score:120.00
pausts           class1:  118  class0:    0  score:118.00
vakcīna          class1:  118  class0:    0  score:118.00
vakcīnas         class1:  460  class0:    3  score:115.00
mrns             class1:  113  class0:    0  score:113.00
sputnik          class1:  112  class0:    0  score:112.00
ru               class1:   99  class0:    0  score:99.00
pildīts          class1:  191  class0:    1  score:95.50
ekrānuzņēmums    class1:   93  class0:    0  score:93.00
difterijas       class1:   92  class0:    

Lexical analysis of full-text content reveals strong thematic and stylistic differences between misinformation-related discourse and reliable news reporting.

Misinformation-related texts (class 1) are dominated by domain-specific and discourse-specific vocabulary. A large proportion of the most distinctive words relate to recurring misinformation topics, particularly health and vaccination (e.g., “vakcīna”, “vakcinācija”, “pfizer”, “mRNS”). In addition, many terms reflect the structure of fact-checking narratives, including references to sources and online content (e.g., “ieraksta”, “ekrānuzņēmums”, “komentāros”), as well as explicit debunking language (e.g., “maldi”, “maldus”).

This suggests that misinformation-related articles are characterized not only by specific topics, but also by a distinct explanatory and refutation-oriented writing style, often referencing claims, sources, and evidence.

In contrast, reliable news texts (class 0) are dominated by geopolitical, economic, and event-driven vocabulary, such as “Irāna”, “triecieni”, “nafta”, and “kodolprogrammas”. These terms reflect factual reporting focused on real-world events, locations, and processes, rather than on analyzing or debunking claims.

Overall, the lexical patterns indicate that misinformation-related texts combine topic-specific content with a characteristic analytical discourse structure, while reliable news emphasizes event-based, descriptive reporting. These findings suggest that effective text-level signal features should capture both topic indicators and discourse-style patterns, rather than relying solely on individual keywords.

Title:“is / isn’t / really” → skepticism
Text: “screenshot / recording / delusion” → fact-checking structure
This means:
TEXT signals ≠ TITLE signalsTEXT signals = discourse patterns, not just words

# Text signal groups

This code checks how often groups of text signals appear in each class, similar to the title analysis.
We validate three groups:
• debunking signals — language of refutation and deception,
• source/reference signals — references to posts, comments, screenshots, etc.,
• topic signals — common misinformation topics that we found in the texts.

Additionally:
1. compare the frequency of these signals by class,
2. also look at them by text length groups,
3. get a simple “feature importance” summary,
4. print examples for manual inspection.

In [ ]:
import pandas as pd
import re

# =========================================================
# 1) LOAD SPLITS + MERGE
# =========================================================
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

df = pd.concat([train_df, val_df, test_df], ignore_index=True)

df["full_text"] = df["full_text"].fillna("").astype(str).str.strip()
df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)

# =========================================================
# 2) TEXT LENGTH
# =========================================================
WORD_REGEX = re.compile(r"\b\w+\b", flags=re.UNICODE)

df["text_len_chars"] = df["full_text"].str.len()
df["text_len_words"] = df["full_text"].apply(lambda x: len(WORD_REGEX.findall(x)))

print("\nAVERAGE TEXT LENGTH BY CLASS\n")
print(df.groupby("class_label")[["text_len_chars", "text_len_words"]].mean())

print("\nMEDIAN TEXT LENGTH BY CLASS\n")
print(df.groupby("class_label")[["text_len_chars", "text_len_words"]].median())

# =========================================================
# 3) TEXT SIGNAL GROUPS
# =========================================================
debunking_patterns = [
    r"\bmaldi\b",
    r"\bmaldus\b",
    r"\bmaldin\w*\b",
    r"\bnepaties\w*\b",
    r"\bmelo\b",
    r"\bmeli\b",
    r"\bnav\s+taisnība\b",
    r"\bpierādījum\w*\b",
    r"\batmasko\w*\b",
]

source_reference_patterns = [
    r"\bierakst\w*\b",
    r"\bekrānuzņēm\w*\b",
    r"\bkomentār\w*\b",
    r"\bpublikāc\w*\b",
    r"\brakst\w*\b",
    r"\bvideo\b",
    r"\bdalījušies\b",
    r"\bpausts\b",
]

topic_patterns = [
    r"\bcovid\b",
    r"\bvakcīn\w*\b",
    r"\bvakcin\w*\b",
    r"\bpotes\b",
    r"\bpfizer\b",
    r"\bmrns\b",
    r"\bsputnik\b",
    r"\bkremļa\b",
    r"\bputina\b",
    r"\bķīmisk\w*\b",
    r"\bpvo\b",
]

debunking_regex = re.compile("|".join(debunking_patterns), flags=re.IGNORECASE | re.UNICODE)
source_reference_regex = re.compile("|".join(source_reference_patterns), flags=re.IGNORECASE | re.UNICODE)
topic_regex = re.compile("|".join(topic_patterns), flags=re.IGNORECASE | re.UNICODE)

df["has_debunking_signal"] = df["full_text"].str.lower().str.contains(debunking_regex, regex=True)
df["has_source_reference_signal"] = df["full_text"].str.lower().str.contains(source_reference_regex, regex=True)
df["has_topic_signal_text"] = df["full_text"].str.lower().str.contains(topic_regex, regex=True)

df["has_any_text_signal"] = (
    df["has_debunking_signal"] |
    df["has_source_reference_signal"] |
    df["has_topic_signal_text"]
)

# =========================================================
# 4) LENGTH GROUPS
# =========================================================
median_len = df["text_len_words"].median()

df["length_group"] = df["text_len_words"].apply(
    lambda x: "shorter_or_equal" if x <= median_len else "longer"
)

print("\nMEDIAN TEXT WORD LENGTH THRESHOLD:", median_len)

# =========================================================
# 5) SHARE BY CLASS
# =========================================================
print("\nSHARE OF TEXT SIGNAL GROUPS BY CLASS\n")
share_by_class = df.groupby("class_label")[[
    "has_debunking_signal",
    "has_source_reference_signal",
    "has_topic_signal_text",
    "has_any_text_signal"
]].mean()
print(share_by_class)

# =========================================================
# 6) SHARE BY CLASS + LENGTH
# =========================================================
print("\nSHARE OF TEXT SIGNAL GROUPS BY CLASS AND LENGTH\n")
share_by_class_length = df.groupby(["class_label", "length_group"])[[
    "has_debunking_signal",
    "has_source_reference_signal",
    "has_topic_signal_text",
    "has_any_text_signal"
]].mean().reset_index()
print(share_by_class_length)

# =========================================================
# 7) COUNTS BY CLASS + LENGTH
# =========================================================
print("\nCOUNTS OF TEXT SIGNAL GROUPS BY CLASS AND LENGTH\n")
counts_by_class_length = df.groupby(["class_label", "length_group"])[[
    "has_debunking_signal",
    "has_source_reference_signal",
    "has_topic_signal_text",
    "has_any_text_signal"
]].sum().reset_index()
print(counts_by_class_length)

# =========================================================
# 8) SIMPLE FEATURE IMPORTANCE VIEW
# =========================================================
print("\nFEATURE IMPORTANCE (RULE-BASED TEXT SIGNALS)\n")

for col in [
    "has_any_text_signal",
    "has_debunking_signal",
    "has_source_reference_signal",
    "has_topic_signal_text",
]:
    class0 = df[df["class_label"] == 0][col].mean()
    class1 = df[df["class_label"] == 1][col].mean()

    diff = class1 - class0
    ratio = (class1 / class0) if class0 > 0 else float("inf")

    print(
        f"{col:28} "
        f"class0:{class0:.3f}  "
        f"class1:{class1:.3f}  "
        f"diff:{diff:.3f}  "
        f"ratio:{ratio:.1f}"
    )

# =========================================================
# 9) EXAMPLES FOR MANUAL CHECK
# =========================================================
print("\nEXAMPLES: DEBUNKING SIGNALS\n")
examples_debunking = df[df["has_debunking_signal"]][
    ["class_label", "full_text", "text_len_words"]
].head(10)
print(examples_debunking.to_string(index=False))

print("\nEXAMPLES: SOURCE / REFERENCE SIGNALS\n")
examples_source = df[df["has_source_reference_signal"]][
    ["class_label", "full_text", "text_len_words"]
].head(10)
print(examples_source.to_string(index=False))

print("\nEXAMPLES: TOPIC SIGNALS\n")
examples_topic = df[df["has_topic_signal_text"]][
    ["class_label", "full_text", "text_len_words"]
].head(10)
print(examples_topic.to_string(index=False))


AVERAGE TEXT LENGTH BY CLASS

             text_len_chars  text_len_words
class_label                                
0               2803.841298      377.145176
1               4198.426931      567.799582

MEDIAN TEXT LENGTH BY CLASS

             text_len_chars  text_len_words
class_label                                
0                    2024.0           268.0
1                    3441.5           470.0

MEDIAN TEXT WORD LENGTH THRESHOLD: 359.0

SHARE OF TEXT SIGNAL GROUPS BY CLASS

             has_debunking_signal  has_source_reference_signal  \
class_label                                                      
0                        0.041479                     0.229035   
1                        0.362213                     0.883090   

             has_topic_signal_text  has_any_text_signal  
class_label                                              
0                         0.073940             0.311993  
1                         0.402923             0.921712  

SHARE OF

The analysis of text-level signal groups reveals a very strong separation between misinformation-related discourse and reliable news articles.

Misinformation-related texts (class 1) exhibit a consistently high presence of all defined signal groups. In particular, source/reference signals appear in 88.3% of class 1 texts, indicating that these articles frequently refer to external content such as social media posts, screenshots, or publications. Debunking signals are present in 36.2% of cases, reflecting the explicit use of refutation language, while topic-related signals occur in 40.3% of texts. Overall, more than 92% of misinformation-related texts contain at least one of the defined signal types.

In contrast, reliable news articles (class 0) show substantially lower signal presence across all categories. Only 4.1% contain debunking signals, 22.9% include source/reference patterns, and 7.4% contain topic-related signals. This confirms that such linguistic patterns are not typical for standard journalistic reporting.

The feature importance analysis highlights that source/reference signals are the most discriminative feature (difference: 0.654), followed by debunking and topic signals. Debunking signals, in particular, show a very high relative difference (ratio: 8.7), indicating that they are strongly associated with misinformation-related discourse.

Importantly, these patterns remain stable across both shorter and longer texts, suggesting that the observed differences are not driven solely by text length but reflect genuine structural and linguistic characteristics of the content.

Overall, the results demonstrate that text-level signals provide a highly informative and interpretable feature space, capable of capturing the distinctive discourse structure of misinformation analysis and fact-checking articles.